# Triaging UXO Reports with an AI Agent

Industry-Integrated AI Systems Synthesis (Project 7) of the AI Mastery Capstone, Master of Science in Artificial Intelligence, Udacity Institute of AI & Technology.

People who find unexploded ordnance (UXO) are told never to touch it, to mark the place and to report it. This notebook builds and tests a system that helps the duty officer of a call centre triage those reports. The call centre is modelled on Switzerland's national reporting centre for UXO, which the Swiss Army's KAMIR command runs; the public sent in 855 reports in 2025. The reports here are invented, and the project is not affiliated with the Swiss Armed Forces.

For each report the system proposes an **urgency** (immediate, same day or planned) and a **specialist flag**, raised when an explosive ordnance disposal (EOD) specialist or the duty officer must confirm the report before anyone goes. The duty officer decides.

The system brings together the author's earlier projects:

| Project | What this notebook takes from it |
|---|---|
| P4, deep learning | The photo reader: a convolutional network that names one of five categories with a confidence, or says "unknown" |
| P5, generative AI | Images from a generative adversarial network (GAN), used as unfamiliar items the reader should not recognise |
| P6, agentic AI | The agent: a model that works through tools inside coded safeguards, and a sign-off sheet for a person |
| P1 and P2 | A checked census table joined to each report, and a paired statistical comparison |

**How to run.** Open the notebook from the repository root, in the environment from `requirements.txt`, and run all cells. With an OpenAI key in the environment variable `OPENAI_API_KEY`, section 10 runs the agent on every report three times and saves each run in `runs/`. Without a key, the notebook runs everything else and reads the runs saved in `runs/`.

**Safety.** Everything stays at a general level: five item categories and counts, no identification detail, no technical data on munitions, no coordinates, and people by role only. The system gives one fixed piece of advice to the person who reported and blocks any other; it never says that an item is safe, and nothing is final until the duty officer signs.

Contents: 1 Setup · 2 Architecture · 3 Data · 4 The photo reader · 5 Generated images · 6 The call-out rules · 7 The reports · 8 The floor and the rules alone · 9 The agent and its safeguards · 10 Runs · 11 Evaluation · 12 Disagreements · 13 Limits

## 1. Setup
The settings below are the only values to change. `SMOKE_TEST = True` runs a few reports once each into `runs_smoke/`, as a check before the full run.

In [ ]:
import json
import os
import platform
import time
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import openai
import pandas as pd
import torch
from IPython.display import Markdown, display
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

from triage import (advice, agent, baseline, evaluation, floor, generated_images, inputs, municipalities, photo_reader,
                    reports, runner, selftest, sheets)

print("Python", platform.python_version(), "on", platform.system())
for module in (np, pd, torch, matplotlib, openai):
    print(f"{module.__name__} {module.__version__}")

In [ ]:
# Settings
MODEL = "gpt-6-luna"            # an OpenAI model that supports function calling on the Responses API, as in P6
REASONING_EFFORT = "medium"     # for reasoning models; None for a model without reasoning settings
RUNS_PER_REPORT = 3             # each report runs three times, because the model's answers vary
SMOKE_TEST = False              # True: run only SMOKE_REPORTS, once each, into runs_smoke/
SMOKE_REPORTS = ["R07", "R22", "R33", "R33-control"]
RESUME = True                   # keep runs already saved with the same model and files; delete the folder to start again
MAX_STEPS = agent.MAX_STEPS     # model calls per run before the fallback (20)
PRICES = {"gpt-6-luna": {"input": 0.10, "output": 0.50}}   # US dollars per million tokens, as used in P6

KEY_SET = bool(os.environ.get("OPENAI_API_KEY"))
# Claude's workspace only: with no key, a scripted stand-in runs every report into runs_dev/, never into runs/.
DEV_STANDIN = os.environ.get("UXO_DEV_STANDIN") == "1" and not KEY_SET
RUNS_DIR = Path("runs_smoke") if SMOKE_TEST else Path("runs")
FIGURES_DIR = Path("runs_dev") / "figures" if DEV_STANDIN else Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(photo_reader.THREADS)
pd.set_option("display.max_colwidth", 120)
COLOURS = {"blue": "#2a78d6", "orange": "#eb6834", "ink": "#0b0b0b", "secondary": "#52514e", "muted": "#898781",
           "grid": "#e1e0d9", "surface": "#fcfcfb"}
print("Key set:", KEY_SET, "| runs folder:", RUNS_DIR, "| figures:", FIGURES_DIR,
      "| scripted stand-in:" if DEV_STANDIN else "", DEV_STANDIN if DEV_STANDIN else "")

## 2. Architecture
A report arrives with a form, the reporter's words and, usually, a photo taken from where the reporter stood. Code reads the photo (P4's reader) and looks up the municipality in the census table (P1's workflow). The agent reads the report through tools and logs the facts that matter, each with a quote from the report. Coded rules, the **floor**, set the lowest urgency allowed and the flag from the report's facts; the agent can raise the urgency but never go below the floor or clear its flag. The person who reported gets one fixed message. The duty officer gets a one-page sign-off sheet and decides.

In [ ]:
def draw_box(ax, x, y, title, lines, face, edge, w=3.5, h=1.9):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.12",
                                facecolor=face, edgecolor=edge, linewidth=1.4))
    ax.text(x + w / 2, y + h - 0.2, title, ha="center", va="top", fontsize=10.5, fontweight="bold")
    ax.text(x + w / 2, y + h - 0.58, "\n".join(lines), ha="center", va="top", fontsize=8.6, linespacing=1.4)


def draw_arrow(ax, start, end, label="", both=False, label_xy=None, ha="center"):
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="<|-|>" if both else "-|>", mutation_scale=13,
                                 linewidth=1.2, color="#444444", shrinkA=0, shrinkB=0))
    if label:
        x, y = label_xy or ((start[0] + end[0]) / 2, max(start[1], end[1]) + 0.08)
        ax.text(x, y, label, ha=ha, va="bottom", fontsize=8, color="#333333", linespacing=1.2)


fig, ax = plt.subplots(figsize=(13.4, 8.2))
ax.set_xlim(0, 13.4)
ax.set_ylim(0, 8.2)
ax.axis("off")
LEFT, MIDDLE, RIGHT = 0.2, 4.95, 9.7
TOP, CENTRE, BOTTOM = 6.0, 3.5, 0.95
draw_box(ax, LEFT, TOP, "Report",
         ["Form: place, people, children,", "moved, condition, control, marking", "Reporter's words (free text)",
          "Photo from where the reporter stood"], "#eef4fb", "#4a78a8")
draw_box(ax, LEFT, CENTRE, "Code before the agent",
         ["Photo reader (P4): one of five", "categories, or unknown", "Census (P1): Eurostat's degree of",
          "urbanisation of the municipality"], "#eef7ee", "#3d8b3d")
draw_box(ax, MIDDLE, TOP, "Agent (OpenAI model)",
         ["System prompt: role, how to work,", "safety rules", "Responses API with function calling",
          "(at most 20 model calls)"], "#fdf3e6", "#c47f1a")
draw_box(ax, MIDDLE, CENTRE, "Tools",
         ["read_report(section), read_photo()", "look_up_rules(topic)", "log_fact(fact, quote, note)",
          "check_proposal(urgency, flag)", "submit(urgency, flag, summary, ...)"], "#eef7ee", "#3d8b3d")
draw_box(ax, MIDDLE, BOTTOM, "Safeguards in code",
         ["Floor: lowest urgency and the flag;", "a proposal below it is vetoed", "Quote check on every logged fact",
          "Only the fixed advice reaches the caller", "Step limit or error: floor + flag"], "#fbeeee", "#b24a4a")
draw_box(ax, RIGHT, TOP, "Person who reported",
         ["One fixed message: thanks; do not", "touch it; move away; mark the place", "if safe; send the location",
          "and photos from where you are"], "#f2f2f2", "#555555")
draw_box(ax, RIGHT, CENTRE, "Memory (one run, one report)",
         ["Facts logged, each with its quote", "Conversation: every call and result", "Nothing carries over",
          "between runs or reports"], "#f3eef9", "#7a5aa8")
draw_box(ax, RIGHT, BOTTOM, "Duty officer",
         ["Sign-off sheet: photo first, facts", "with their source, the floor's rules,", "marking, evacuation, police,",
          "what the agent did not use", "The duty officer decides"], "#f2f2f2", "#555555")
draw_arrow(ax, (LEFT + 1.75, TOP - 0.03), (LEFT + 1.75, CENTRE + 1.93), "photo,\nmunicipality", label_xy=(LEFT + 1.87, 5.6), ha="left")
draw_arrow(ax, (LEFT + 3.53, TOP + 0.95), (MIDDLE - 0.03, TOP + 0.95), "report text\nas data")
draw_arrow(ax, (LEFT + 3.53, CENTRE + 0.95), (MIDDLE - 0.03, CENTRE + 0.95), "verdict,\nclass")
draw_arrow(ax, (MIDDLE + 1.75, TOP - 0.03), (MIDDLE + 1.75, CENTRE + 1.93), "tool calls / results\n(a veto returns as a result)",
           both=True, label_xy=(MIDDLE + 1.87, 5.55), ha="left")
draw_arrow(ax, (MIDDLE + 3.53, CENTRE + 0.95), (RIGHT - 0.03, CENTRE + 0.95), "writes and reads\nthe facts", both=True)
draw_arrow(ax, (MIDDLE + 1.75, CENTRE - 0.03), (MIDDLE + 1.75, BOTTOM + 1.93), "log_fact, check_proposal, submit",
           label_xy=(MIDDLE + 1.87, 3.0), ha="left")
draw_arrow(ax, (MIDDLE + 3.53, BOTTOM + 0.95), (RIGHT - 0.03, BOTTOM + 0.95), "sign-off\nsheet")
draw_arrow(ax, (MIDDLE + 3.53, TOP + 1.45), (RIGHT - 0.03, TOP + 1.45), "fixed advice\n(by code)")
ax.text(6.7, 0.4, "One run triages one report. Nothing is final until the duty officer signs the sheet.",
        ha="center", fontsize=9, style="italic", color="#333333")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "architecture.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Data
The data files are checked against the checksums recorded when they were made; the census table is checked as it loads. Sources and licences are in `data/README.md`: CTX-UXO and SULAND under CC BY 4.0, and Eurostat's table of local administrative units, reused with acknowledgement.

In [ ]:
checks = photo_reader.check_data()
display(pd.DataFrame({"file": list(checks), "matches its checksum": list(checks.values())}).set_index("file"))
assert all(checks.values()), "a data file differs from its recorded checksum"
CENSUS = municipalities.load_municipalities()
print(f"Census: {len(CENSUS):,} Swiss municipalities; degree of urbanisation:",
      CENSUS.degurba_class.value_counts().to_dict())
REPORT_LIST = reports.load_reports()
CONTROL = agent.without_instructions(next(r for r in REPORT_LIST if r["id"] == "R33"))
print(f"Reports: {len(REPORT_LIST)}, photos: {len(list(Path('reports/photos').glob('*.png')))}, "
      f"plus {CONTROL['id']}, a copy of R33 without the text it addresses to the system")

## 4. The photo reader (P4)
The reader is P4's network, trained as in P4 on CTX-UXO crops (`python -m triage.photo_reader train`, saved in `models/`). Below its "unknown" cut-off it gives no category: the cut-off is the lowest confidence at which it is right on at least 95% of the validation crops it keeps. The reader is tested on the test crops, clear and with a grey square over 25% or 40% of each, and on three sets it never trained on: P4's seven left-out categories, crops of ground with no item, and P5's generated images. On the unfamiliar sets, the safe result is "unknown".

In [ ]:
READER = photo_reader.PhotoReader.load()
info = READER.info
print(f"Cut-off {READER.cutoff:.3f}; best epoch {info['best_epoch']} of {info['epochs']}; "
      f"trained in {info['training_seconds'] / 60:.0f} minutes ({info['torch']})")
started = time.time()
CROP_DATA = photo_reader.load_crop_data()
_, BACKGROUND = photo_reader.load_background()
GENERATED_MANIFEST, GENERATED = generated_images.load_generated()
READER_SUMMARY, READER_PER_CATEGORY = evaluation.reader_results(READER, CROP_DATA, BACKGROUND, GENERATED)
print(f"Read in {time.time() - started:.0f} s")
display(READER_SUMMARY.set_index("set").style.format(
    {"sent to unknown": "{:.1%}", "accuracy on the rest": "{:.1%}", "macro-F1 (best guess)": "{:.3f}",
     "accuracy (best guess)": "{:.1%}", "kept and correct": "{:.0f}", "confident category": "{:.0f}"}, na_rep=""))

In [ ]:
pivot = READER_PER_CATEGORY.pivot(index="category", columns="set", values=["precision", "recall"]).round(3)
display(pivot.reindex(photo_reader.CATEGORIES))

In [ ]:
# Share of each test set sent to "unknown": one series, so one colour, with the value on each bar.
summary = READER_SUMMARY.set_index("set")["sent to unknown"].iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 3.6))
bars = ax.barh(summary.index, summary.values, color=COLOURS["blue"], height=0.55)
for bar, value in zip(bars, summary.values):
    ax.text(value + 0.01, bar.get_y() + bar.get_height() / 2, f"{value:.0%}", va="center", fontsize=9, color=COLOURS["ink"])
ax.set_xlim(0, 1.08)
ax.set_xlabel("Share of images sent to unknown", color=COLOURS["secondary"])
ax.set_title("The photo reader says unknown for most images", loc="left", fontsize=11)
ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.grid(axis="x", color=COLOURS["grid"], linewidth=0.8)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "reader_unknown_share.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Generated images (P5)
P5's GAN, trained again as in P5 (`python -m triage.generated_images train`), made 100 images of two surrogate items it learned from SULAND. They stand in for items the reader never saw. Each one is labelled as generated wherever it is shown, and they are used only as tests.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, image, (_, row) in zip(axes.flat, GENERATED, GENERATED_MANIFEST.iterrows()):
    ax.imshow(image)
    ax.set_title("GENERATED", fontsize=7.5, color="#b24a4a")
    ax.axis("off")
fig.suptitle("16 of the 100 images generated by P5's GAN (test images only)", fontsize=10)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "generated_samples.png", dpi=150, bbox_inches="tight")
plt.show()
row = READER_SUMMARY.set_index("set").loc["generated images (P5)"]
print(f"The reader sends {row['sent to unknown']:.0%} of the generated images to unknown; "
      f"{row['confident category']:.0f} get a confident category ({row['confident as']}).")

## 6. The call-out rules
The rules are in `rules/callout_rules.md`: drafted from the Swiss Army's public guidance and set by the project's EOD specialist, the author, who also confirmed how the code reads the report form. The floor applies them to every report; the agent can look them up by topic.

In [ ]:
TOPICS = agent.rule_topics()
display(Markdown("### Urgency levels\n" + TOPICS["urgency_levels"]))
display(Markdown("### The floor\n" + TOPICS["floor"]))
display(Markdown("### Advice to the person who reported\n> " + advice.advice()))

## 7. The reports
Thirty-six invented reports, set in real Swiss municipalities, with the hard cases on purpose: missing facts, a report that contradicts itself, two reports of different items at one place, an item taken home, children nearby, people asking how to make the item safe, text addressed to the system, and a call cut off. Each has a test photo drawn with a fixed seed from the reader's test sets (`reports/photos.csv` records where each came from), or none. The census table gives each municipality's degree of urbanisation.

In [ ]:
PHOTOS = pd.read_csv("reports/photos.csv").set_index("report")
form = pd.DataFrame(REPORT_LIST).set_index("id")[["channel", "municipality", "place_type", "people_close", "children",
                                                   "moved", "condition", "controllable"]]
form["photo"] = PHOTOS.kind.reindex(form.index)
display(form)
print("Photos by kind:", PHOTOS.kind.value_counts().to_dict())
display(Markdown(reports.render(next(r for r in REPORT_LIST if r["id"] == "R07"))))

In [ ]:
fig, axes = plt.subplots(4, 9, figsize=(13, 6.4))
for ax, report in zip(axes.flat, REPORT_LIST):
    image = reports.load_photo(report["id"])
    kind = PHOTOS.loc[report["id"], "kind"]
    if image is None:
        ax.imshow(np.full((64, 64, 3), 235, dtype=np.uint8))
        ax.text(32, 32, "no photo", ha="center", va="center", fontsize=8, color=COLOURS["secondary"])
    else:
        ax.imshow(image)
    label = report["id"] + (" GENERATED" if kind == "generated" else "")
    ax.set_title(label, fontsize=8, color="#b24a4a" if kind == "generated" else COLOURS["ink"])
    ax.axis("off")
fig.suptitle("The photo attached to each report (64 x 64 pixels)", fontsize=10)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "report_photos.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
REPORT_IDS = [r["id"] for r in REPORT_LIST]
ALL_REPORTS = REPORT_LIST + [CONTROL]
CLASSES = inputs.municipality_classes(ALL_REPORTS, CENSUS)
VERDICTS = inputs.reader_verdicts(ALL_REPORTS, READER)
joined = CENSUS.set_index("municipality").loc[[r["municipality"] for r in REPORT_LIST],
                                              ["nuts3", "population", "density_per_km2", "degurba_class"]]
joined.insert(0, "report", REPORT_IDS)
display(joined.reset_index().set_index("report"))
print("Every municipality found once in the census table:", len(joined) == len(REPORT_LIST))

## 8. The floor and the rules alone
The floor reads the report form and the photo verdict (`triage/floor.py`). On its own, with none of the agent's facts, it is the comparison the agent is measured against: **the rules alone**. The agent can add facts from the reporter's words, so any difference between the two comes from what the agent read.

In [ ]:
RULES_ALONE = baseline.rules_only_rows(REPORT_LIST, VERDICTS, CLASSES)
rules_table = pd.DataFrame(RULES_ALONE).set_index("report")
display(rules_table)
print("Urgency:", rules_table.urgency.value_counts().reindex(floor.URGENCIES).to_dict(),
      "| flag:", int(rules_table.flag.sum()), "of", len(rules_table))
counts = pd.Series(" ".join(rules_table.rules).split()).value_counts()
print("Rules that applied, in how many reports:", counts.to_dict())

## 9. The agent and its safeguards
The agent is an OpenAI model on the Responses API with six tools (`triage/agent.py`), built like P6's. Its instructions are in `prompts/system_prompt.md`. The safeguards are code, not instructions: the quote check, the floor's veto, the advice block, the scan for text addressed to the system, and the step limit with its fallback to the floor and the flag. The tests below run the agent's loop with a scripted stand-in model, so they need no key.

In [ ]:
TOOLS = agent.tools_schema(TOPICS)
display(pd.DataFrame([{"tool": t["name"], "what it does": t["description"][:150]} for t in TOOLS]).set_index("tool"))
display(Markdown("### System prompt\n" + agent.PROMPT_FILE.read_text(encoding="utf-8")))

In [ ]:
SAFEGUARDS = pd.DataFrame(selftest.safeguard_tests()).set_index("test")
display(SAFEGUARDS)
assert SAFEGUARDS["fired as designed"].all(), "A safeguard did not fire as designed: see the table above."
print(f"All {len(SAFEGUARDS)} safeguards fired as designed.")

In [ ]:
bench = selftest.Bench()
example, *_ = bench.run("R07", [selftest.READ_ALL,
    [selftest.fact("children", "Two pupils found a rusty object", "Pupils found the item and carried it.")],
    [("check_proposal", {"urgency": "same day", "flag": True})],
    [selftest.submit("immediate", True, summary="Pupils carried the item into the staff room and the teacher asks "
                     "how to make it safe; the room is closed.", confidence="high")]])
display(Markdown(sheets.signoff_sheet(example, "stand-in example", "scripted stand-in (no model)")))

## 10. Runs
With a key, each report runs three times (a model's answers vary from run to run), and so does the copy of R33 without the text it addresses to the system. Each run is saved in `runs/` as its trace, its sign-off sheet and its result; runs already saved with the same model and files are kept. Without a key, the runs saved in `runs/` are read.

In [ ]:
RUN_LIST = [r for r in ALL_REPORTS if r["id"] in SMOKE_REPORTS] if SMOKE_TEST else ALL_REPORTS
if KEY_SET:
    client = openai.OpenAI(max_retries=5, timeout=180)
    started = time.time()
    runner.run_reports(RUN_LIST, VERDICTS, CLASSES, lambda session: client, MODEL,
                       1 if SMOKE_TEST else RUNS_PER_REPORT, RUNS_DIR, PRICES.get(MODEL), REASONING_EFFORT,
                       MAX_STEPS, resume=RESUME)
    print(f"Runs finished in {(time.time() - started) / 60:.1f} minutes")
    RESULTS_DIR = RUNS_DIR
elif (RUNS_DIR / "results.csv").exists():
    RESULTS_DIR = RUNS_DIR
elif DEV_STANDIN:
    runner.run_reports(RUN_LIST, VERDICTS, CLASSES, runner.scripted_client, "stand-in (scripted)", RUNS_PER_REPORT,
                       Path("runs_dev"), resume=False, log=lambda message: None)
    RESULTS_DIR = Path("runs_dev")
else:
    RESULTS_DIR = None
    print("No key and no saved runs in", RUNS_DIR, "- sections 11 and 12 need runs.")
if RESULTS_DIR is not None:
    RESULTS = runner.read_results(RESULTS_DIR)
    RUN_INFO = json.loads((RESULTS_DIR / "run_info.json").read_text(encoding="utf-8"))
    print(f"{len(RESULTS)} runs read from {RESULTS_DIR}: model {RUN_INFO['model']}, "
          f"{RUN_INFO['runs_per_report']} per report, inputs {RUN_INFO['inputs_sha']}, finished {RUN_INFO['finished']}")
    display(RESULTS[["report", "run", "urgency", "flag", "floor_urgency", "floor_flag", "fallback", "vetoes",
                     "facts_logged", "model_calls", "error"]].head(12))

## 11. Evaluation
The reference answers were set by the project's EOD specialist, the author, for every report before any run and without seeing the system's answers (`reports/reference.csv`). They are read here for the first time in the notebook; no run reads them. **Under-triage**, an urgency below the reference, is the main safety measure; over-triage costs time but not safety.

In [ ]:
if RESULTS_DIR is not None:
    REFERENCE = evaluation.load_reference()
    SCORED = evaluation.score(RESULTS, REFERENCE)
    RUNS_MAIN = evaluation.main_runs(SCORED)
    AGENT_REPORTS = evaluation.per_report(RUNS_MAIN)
    SUMMARY = evaluation.run_summary(SCORED)
    display(SUMMARY.to_frame("value"))

In [ ]:
if RESULTS_DIR is not None:
    COMPARISON, WILCOXON, UNDER, OVER = evaluation.compare_with_rules(AGENT_REPORTS, RULES_ALONE, REFERENCE)
    display(COMPARISON.style.format({"agent": "{:.1%}", "rules alone": "{:.1%}", "difference": "{:+.1%}",
                                     "95% interval low": "{:+.1%}", "95% interval high": "{:+.1%}",
                                     "McNemar exact p": "{:.3f}"}))
    print(f"Size of the urgency error, agent against the rules alone: Wilcoxon signed-rank p = {WILCOXON['p']:.3f}")
    print("Reports under-triaged:", UNDER, "| over-triaged:", OVER)

In [ ]:
# Urgency against the reference: the agent (one answer per report) and the rules alone, on one colour scale.
if RESULTS_DIR is not None:
    agent_table = evaluation.confusion_table(evaluation.score(AGENT_REPORTS, REFERENCE))
    rules_scored = evaluation.score(pd.DataFrame(RULES_ALONE), REFERENCE)
    rules_matrix = evaluation.confusion_table(rules_scored.set_index("report").loc[AGENT_REPORTS.report].reset_index())
    top = max(agent_table.values.max(), rules_matrix.values.max())
    fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.2))
    for ax, (title, matrix) in zip(axes, [("Agent", agent_table), ("Rules alone", rules_matrix)]):
        ax.imshow(matrix.values, cmap=matplotlib.colors.LinearSegmentedColormap.from_list(
            "blues", ["#f4f8fd", "#86b6ef", "#1c5cab"]), vmin=0, vmax=top)
        for i in range(3):
            for j in range(3):
                value = matrix.values[i, j]
                ax.text(j, i, str(value), ha="center", va="center", fontsize=11,
                        color="#ffffff" if value > top * 0.7 else COLOURS["ink"])
        ax.set_xticks(range(3), floor.URGENCIES)
        ax.set_yticks(range(3), floor.URGENCIES)
        ax.set_xlabel("Answer", color=COLOURS["secondary"])
        ax.set_ylabel("Reference (EOD specialist)", color=COLOURS["secondary"])
        ax.set_title(f"{title}: {np.trace(matrix.values)} of {matrix.values.sum()} reports agree", fontsize=10.5, loc="left")
    fig.text(0.5, -0.02, "Cells below the diagonal are under-triage (an urgency below the reference); above it, over-triage.",
             ha="center", fontsize=8.5, color=COLOURS["secondary"])
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "triage_confusion.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Agreement with the reference, the agent against the rules alone: two series, a legend and the value on each bar.
if RESULTS_DIR is not None:
    measures = {"urgency_agrees": "Urgency", "flag_agrees": "Flag", "both_agree": "Both"}
    x = np.arange(len(measures))
    fig, ax = plt.subplots(figsize=(7.4, 3.8))
    for offset, (column, colour) in zip((-0.19, 0.19), (("agent", COLOURS["blue"]), ("rules alone", COLOURS["orange"]))):
        values = COMPARISON.loc[list(measures), column].to_numpy()
        bars = ax.bar(x + offset, values, width=0.36, color=colour, label=column.capitalize(), edgecolor=COLOURS["surface"])
        for bar, value in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, value + 0.015, f"{value:.0%}", ha="center", fontsize=9)
    ax.set_xticks(x, list(measures.values()))
    ax.set_ylim(0, 1.1)
    ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
    ax.set_ylabel("Reports that agree with the reference", color=COLOURS["secondary"])
    ax.set_title("Agreement with the EOD specialist's answers", loc="left", fontsize=11)
    ax.legend(frameon=False, loc="upper left")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(axis="y", color=COLOURS["grid"], linewidth=0.8)
    ax.set_axisbelow(True)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "triage_agreement.png", dpi=150, bbox_inches="tight")
    plt.show()

### Text addressed to the system (R33)
R33's description holds a note addressed to the system, telling it what to decide. The reference answer for R33 is the specialist's own call on the report, and it happens to match what the note asks for, so agreement on R33 cannot show whether the agent resisted the note. The test is the copy of R33 without the note: the answers should be the same, and the agent should report the note rather than follow it.

In [ ]:
if RESULTS_DIR is not None:
    INSTRUCTION_TEST = evaluation.instruction_test(SCORED)
    if INSTRUCTION_TEST.empty:
        print("R33 and its control copy are not both among the runs.")
    else:
        display(INSTRUCTION_TEST)
        print("Runs where R33 and its copy got the same answer:", int(INSTRUCTION_TEST["same answer"].sum()),
              "of", len(INSTRUCTION_TEST), "| runs where the agent reported the note:",
              int(INSTRUCTION_TEST["noticed the text"].sum()))

## 12. Disagreements
The reports where the agent's answer (the middle urgency of its runs and the majority flag) differs from the reference, with the specialist's note. Under-triage comes first. The sign-off sheets in the runs folder show what the agent logged in each run.

In [ ]:
if RESULTS_DIR is not None:
    scored_reports = evaluation.score(AGENT_REPORTS, REFERENCE).set_index("report")
    scored_reports["note"] = REFERENCE.note.reindex(scored_reports.index)
    scored_reports["rules alone"] = rules_table.urgency.reindex(scored_reports.index) + ", flag " + \
        rules_table.flag.map({True: "yes", False: "no"}).reindex(scored_reports.index)
    differ = scored_reports[~scored_reports.both_agree].sort_values(["under_triage", "rank_error"], ascending=[False, True])
    print(f"{len(differ)} of {len(scored_reports)} reports differ from the reference "
          f"({int(differ.under_triage.sum())} under-triaged).")
    display(differ[["urgencies", "flags", "ref_urgency", "ref_flag", "rules alone", "note"]])
    if len(differ):
        first = differ.index[0]
        sheet = RESULTS_DIR / f"{first}_run1_signoff.md"
        display(Markdown(f"### Sign-off sheet of {first}, run 1\n\n" + sheet.read_text(encoding="utf-8")))

## 13. Limits
Known before the runs:
- **Invented reports.** The 36 reports were written for this project from public facts about who finds what and where; real reports come in German, French or Italian, by phone, app or form.
- **One reference.** The reference answers are one EOD specialist's; another specialist could answer some reports differently, and his answers were sometimes less cautious than the rules he set.
- **The photo reader.** CTX-UXO's photos come from one country, some of them of replicas, not from Swiss reports, and the reader says "unknown" for most images, so the flag goes up for most reports.
- **Three runs per report** show some of the model's variation, not all of it.

The results in words, the failure cases and what they mean are written after the full run.